In [ ]:
!pip install pandas numpy torch transformers datasets peft trl accelerate python-box liger-kernel
!pip uninstall -y bitsandbytes
!pip install --no-cache-dir bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 165.2 MB/s eta 0:00:00


In [ ]:
import os, re, sys, json, string, csv, gzip
import pandas as pd
import numpy as np

from box import Box

import warnings
import datasets
import torch
import peft
import trl
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

from peft import get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from trl import apply_chat_template
import argparse
import logging
import time

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

warnings.filterwarnings("ignore", message="torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly")
os.environ["WANDB_DISABLED"] = "true"
sys.argv = ["qlora_finetuning_script", "--config_path", "/content/drive/MyDrive/CAISI_phase2/qlora-finetuning/configs/cf_files/llama/finetune.yaml", "--save_folder", "/content/drive/MyDrive/CAISI_phase2/qlora-finetuning/4bit-checkpoints/"]

parser = argparse.ArgumentParser()
parser.add_argument('--config_path', help='path to config file', required=True)
# parser.add_argument('--log_file', required=True, help='Path to file to write logs')
# parser.add_argument('--random_state', default=5, help="random state", required=False)
parser.add_argument('--save_folder', help='path to save outputs and log files')


def convert_to_template_from_alpaca(example):
    return {"prompt": [
                         {"role": "user", "content": example['instruction'] + ": " + example['input']}],
            "completion": [
                         {"role": "assistant", "content": example['output']}
                        ]
           }

TEMPLATE_FUNC_MAP = {
    'alpaca': convert_to_template_from_alpaca
}

class QloraFT:
    def __init__(self, config, save_folder):
        self.config = config
        self.save_folder = save_folder

        self.model, self.tokenizer = self.load_model_for_training()

        self.dataset = self.load_dataset_for_training()


    def load_model_for_training(self):
        config = self.config
        logging.info("Loading base model and tokenizer")
        quantization_config=None
        # load_in_8bit=True
        if config.load_in_4bit:
            quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_storage=torch.float16
                )
            # quantization_config = bnb_config
            # load_in_8bit = False


        model = AutoModelForCausalLM.from_pretrained(config.base_model_path, quantization_config=quantization_config, \
               torch_dtype=torch.float16, attn_implementation=config.attn_implementation)
        # model = prepare_model_for_kbit_training(model)

        tokenizer = AutoTokenizer.from_pretrained(config.base_model_path)
        tokenizer.pad_token = tokenizer.eos_token

        return model, tokenizer

    def load_dataset_for_training(self):
        logging.info("loading dataset")
        config = self.config

        # data_path = config.data_path
        data_format = config.data_template

        dataset = load_dataset(config.data_path)
        logging.info("Dataset info: {} ".format(dataset['train'].info))
        og_cols = list(dataset['train'].column_names)

        template_func = TEMPLATE_FUNC_MAP[data_format] # add others later

        # assume only train split
        split_dataset = dataset['train'].train_test_split(test_size=0.15)
        dataset['train'] = split_dataset['train']
        dataset['val'] = split_dataset['test']

        logging.info("Splits train and val with {} and {} instances".format(len(dataset['train']), len(dataset['val'])))

        logging.info("Standardizing to prompt-completion format for TRL")

        for k in dataset:
            dataset[k] = dataset[k].map(template_func, remove_columns=og_cols)
        return dataset

    def run_peft_training(self):
        logging.info("Preparing PEFT model")
        config = self.config
        peft_config = LoraConfig(
                r = config.lora_rank,
                lora_alpha=config.lora_alpha,
                lora_dropout=config.lora_dropout,
                bias="none",
                target_modules=config.target_modules,
                task_type='CAUSAL_LM'
            )

        # model = get_peft_model(self.model, peft_config)
        dataset = self.dataset
        tokenizer = self.tokenizer
        model = self.model

        # a,b = model.get_nb_trainable_parameters()
        # logging.info("{} trainable parameters | ratio to overall: {}".format(a, a/b))

        training_args = SFTConfig(
                # dataset_text_field='messages',
                learning_rate=float(config.learning_rate),
                optim=config.optim,
                use_liger_kernel=config.use_liger_kernel,
                gradient_checkpointing=True,
                ddp_find_unused_parameters=False,
                num_train_epochs=config.num_train_epochs,
                per_device_train_batch_size=config.per_device_train_batch_size,
                gradient_accumulation_steps=config.gradient_accumulation_steps,
                warmup_ratio=config.warmup_ratio,
                lr_scheduler_type=config.lr_scheduler_type,
                max_grad_norm=config.max_grad_norm,
                fp16=True,
                max_length=config.max_seq_length,
                packing=config.use_packing,
                output_dir=self.save_folder,
                save_strategy=config.save_strategy,
                save_only_model=False,
                eval_strategy=config.eval_strategy,
                eval_steps=config.eval_steps if config.eval_strategy=='steps' else None,
                logging_dir=self.save_folder,
                logging_first_step=True,
                logging_strategy=config.logging_strategy,
                logging_steps=config.logging_steps,
                log_level='info',
                seed=config.random_state,
                completion_only_loss=config.completion_only_loss
            )

        # use_cache=True
        # use_reentrant = False
        # gradient_checkpointing=False
        if config.use_gradient_checkpointing:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
            use_cache = False
            # use_reentrant = True
            # gradient_checkpointing = True

        model.config.use_cache = use_cache
        # training_args.gradient_checkpointing = gradient_checkpointing
        # training_args.gradient_checkpointing_kwargs = {"use_reentrant": use_reentrant}

        trainer = SFTTrainer(
                model = model,
                # tokenizer=tokenizer,
                # data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
                args=training_args,
                peft_config=peft_config,
                train_dataset=dataset['train'],
                eval_dataset=dataset['val'],
                # formatting_func=formatting_prompts_func
                processing_class=tokenizer,
            )

        trainer.accelerator.print(f"{trainer.model}")
        if hasattr(trainer.model, "print_trainable_parameters"):
            trainer.model.print_trainable_parameters()

        trainer.train()

        if trainer.is_fsdp_enabled:
            trainer.accelerator.state.fsdp_plugin.set_state_dict_type("FULL_STATE_DICT")
        trainer.model.config.use_cache = True
        # save final adapter
        trainer.save_model(output_dir = self.save_folder)

        training_args.distributed_state.wait_for_everyone()
        tokenizer.save_pretrained(training_args.output_dir)
        logging.info(f'Tokenizer saved to {training_args.output_dir}')


        logging.info("Finished training | final model saved.")


def main(config_path, save_folder):
    config = Box.from_yaml(filename=config_path)

    logging.info("Read config from {}".format(config_path))
    logging.info("Config: {}".format(config))
    # logging.info("random state: {}".format(random_state))
    config.to_yaml(filename=os.path.join(save_folder, "config.yaml"))
    logging.info("Init Model and Dataset..")

    tuner = QloraFT(config=config, save_folder=save_folder)
    # dataset = tuner.load_data()
    logging.info("Starting training at {}".format(time.time()))

    tuner.run_peft_training()

    logging.info("Job done at {}".format(time.time()))


if __name__ == '__main__':
    args = parser.parse_args()
    save_folder = args.save_folder
    os.makedirs(save_folder, exist_ok=True)

    # log_file = os.path.join(save_folder, 'log.txt') # goes to slurm log
    logging.basicConfig(encoding='utf-8', level=logging.DEBUG, \
                        format='%(asctime)s %(message)s')

    main(args.config_path, save_folder)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Using auto half precision backend
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

***** Running training *****
  Num examples = 4,250
  Num Epochs = 5
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 1,330
  Number of trainable parameters = 41,943,040
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is 

Epoch,Training Loss,Validation Loss,Num Tokens
1,0.977000,0.932397,820772.000000
2,0.832300,0.962738,1641544.000000
3,0.536700,1.106187,2462316.000000
4,0.276300,1.306365,3283088.000000
5,0.153400,1.569175,4103860.000000


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/